In [ ]:
# =====================================================
# STEP 0: Mount Google Drive
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

import os
# Change this path to where you have your video in Drive
os.chdir("/content/drive/MyDrive/")  # adjust according to your folder

# =====================================================
# STEP 1: Extract frames from video
# =====================================================
import cv2
import numpy as np

VIDEO_FILE = 'video.mp4'   # <-- put your file here
FPS_OBJETIVO = 24
SECONDS = 10
TOTAL_FRAMES = FPS_OBJETIVO * SECONDS   # = 240
WIDTH = 320
HEIGHT  = 240

vidcap = cv2.VideoCapture(VIDEO_FILE)
fps_original = vidcap.get(cv2.CAP_PROP_FPS)
total_frames_video = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Original video FPS: {fps_original}")
print(f"Total frames in video: {total_frames_video}")
print(f"Duration: {total_frames_video/fps_original:.2f} seconds")

# We calculate which frames to take to get exactly 24 FPS for 10s
# even if the original video has a different FPS
indexes_to_take = np.linspace(0, min(total_frames_video - 1,
                               int(fps_original * SECONDS) - 1),
                               TOTAL_FRAMES, dtype=int)

frames_rgb565 = []

for i, frame_idx in enumerate(indexes_to_take):
    vidcap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    success, frame = vidcap.read()
    if not success:
        print(f"Error reading frame {frame_idx}, using black frame")
        frame = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

    # Resize to 320x240
    frame_resized = cv2.resize(frame, (WIDTH, HEIGHT))

    # OpenCV uses BGR, convert to RGB
    frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)

    # Convert to RGB565 (16 bits)
    # R: bits 15-11 (5 bits), G: bits 10-5 (6 bits), B: bits 4-0 (5 bits)
    r = (frame_rgb[:, :, 0].astype(np.uint16) >> 3) & 0x1F
    g = (frame_rgb[:, :, 1].astype(np.uint16) >> 2) & 0x3F
    b = (frame_rgb[:, :, 2].astype(np.uint16) >> 3) & 0x1F
    rgb565 = (r << 11) | (g << 5) | b   # each value is uint16

    frames_rgb565.append(rgb565)

    if i % 24 == 0:
        print(f"  Processed frame {i+1}/{TOTAL_FRAMES}")

vidcap.release()
print(f"\nTotal frames processed: {len(frames_rgb565)}")
print(f"Estimated total memory: {len(frames_rgb565) * WIDTH * HEIGHT * 2 / 1024 / 1024:.1f} MB")

# =====================================================
# STEP 2: Generate the Assembly file
# IMPORTANT: writing BACKWARDS
# Last frame first, first frame last
# This way, when reading .data sequentially, it plays in correct order
# =====================================================
print("\nGenerating video_data.s file ...")

OUTPUT_FILE = "video_data.s"

with open(OUTPUT_FILE, "w") as f:
    f.write(".data\n")
    f.write(".global video_data_start\n")
    f.write(".global video_data_end\n\n")
    f.write("video_data_start:\n")

    # Frames in normal order (the code reads them in reverse with the pointer)
    for i in range(TOTAL_FRAMES):
        frame = frames_rgb565[i]
        for row in range(HEIGHT):
            for col in range(WIDTH):
                p = int(frame[row, col])
                f.write(f"     .word  0x{p:04X}\n")

    f.write("\nvideo_data_end:\n")
    f.write("     .word  0x0000\n")

print(f"File generated: {OUTPUT_FILE}")
print(f"Approximate .s size: {os.path.getsize(OUTPUT_FILE)/1024/1024:.1f} MB")
print("\nDone! Copy video_data.s to your project in VS Code.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original video FPS: 29.426751592356688
Total frames in video: 308
Duration: 10.47 seconds
  Processed frame 1/240
  Processed frame 25/240
  Processed frame 49/240
  Processed frame 73/240
  Processed frame 97/240
  Processed frame 121/240
  Processed frame 145/240
  Processed frame 169/240
  Processed frame 193/240
  Processed frame 217/240

Total frames processed: 240
Estimated total memory: 35.2 MB

Generating video_data.s file ...
File generated: video_data.s
Approximate .s size: 334.0 MB

Done! Copy video_data.s to your project in VS Code.
